# EmbedMed

## Setup

# ------ 

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

from controllers.ProcessController import ProcessController

config = {
    "GROQ_API_KEY": os.getenv("GROQ_API_KEY"),
    "GENERATION_MODEL": "openai/gpt-oss-120b",
    "EMBEDDING_MODEL": "BAAI/bge-small-en-v1.5",
    "VECTOR_DB_PATH": "chroma_db",
}

controller = ProcessController(config)


In [7]:
controller.load_pdf(
    pdf_path="assets/hypertension-in-adults-diagnosis-and-management.pdf",
    document_id="NICE-NG136-2026",
    title="Hypertension in Adults: Diagnosis and Management",
    version="NG136",
    publication_date="2019-08-28"
)

controller.chunk_documents(document_id="NICE-NG136-2026")

controller.build_vectorstore(collection_name="hypertension_clinical_kb")

print("Pipeline completed successfully")
print(f"Pages loaded: {len(controller.pages)}")
print(f"Chunks created: {len(controller.chunks)}")

Pipeline completed successfully
Pages loaded: 52
Chunks created: 156


In [8]:
result = controller.ask("What is the target blood pressure for people with type 2 diabetes?")
print(result["answer"])
print("\nSources:")
for s in result["retrieved_sources"]:
    print(s)

The current NICE guideline sets a clinic blood‑pressure target of **below 140 mm Hg systolic and below 90 mm Hg diastolic** for people under 80 years of age who have hypertension, whether or not they have type 2 diabetes [Document ID | p. 14 | NICE‑NG136‑2026‑CH‑038]. The committee concluded that there is no evidence to support a different (lower) target specifically for people with type 2 diabetes, and therefore the same <140/90 mm Hg target applies [Document ID | p. 38 | NICE‑NG136‑2026‑CH‑113].

Educational information only; not a diagnosis or medical advice.

Sources:
{'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-113', 'preview': 'on people already receiving treatment and that it lacked information on adverse events.  The committee agreed that there was no evidence to suggest that blood pressure targets  sho'}
{'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-113', 'preview': 'on people already receiving treatment and th

In [9]:
import sys
sys.path.append("evaluation")

from evaluation.eval_questions import EVAL_QUESTIONS
from evaluation.run_evaluation import run_retrieval

sample_results = run_retrieval(controller, EVAL_QUESTIONS[:2], k=3)

for r in sample_results:
    print("Question:", r["question"])
    for chunk in r["retrieved_chunks"]:
        print(f"  [{chunk['chunk_id']}] score={chunk['score']} page={chunk['page']}")
        print(f"  {chunk['chunk_text'][:100]}...")
    print()

AttributeError: 'ChromaProvider' object has no attribute 'search_with_scores'